# Previsao da qualidade do ar com Flower e Pumba

Dataset: Sonawani, Shilpa; Patil, Kailas (2021), “Delhi Multi-Site Air-Quality Data Set”, Mendeley Data, V1, doi: 10.17632/bzhzr9b64v.1.

O programa usa uma estacao por cliente para prever PM2.5 e PM10. Execute em ordem:
**implementacao → hands-on com resultados → discussao e conclusoes → limpeza**.

## 1. Implementacao

Esta parte define os requisitos, a preparacao dos dados, os modelos, o cliente,
o servidor, o caos de rede e as funcoes de execucao e apresentacao.
Nenhum experimento e iniciado nesta parte.

### Pre-requisitos

Use Docker com Compose v2 e instale `requirements.txt` no kernel (`pip install -r requirements.txt`). Os seis CSVs devem estar em `data/`. Execute as células em ordem para construir o modelo, configurar os containers e realizar os experimentos.

Importa as bibliotecas, localiza o tutorial e define os arquivos, os dois alvos e as janelas de 10 horas de entrada e previsão. Confere a presença das seis estações para manter o limite de clientes coerente com os dados.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import math
import subprocess
import tempfile
import time

cwd = Path.cwd()
candidates = (cwd, cwd / "tutorial-fl-pumba", cwd.parent / "tutorial-fl-pumba")
TUTORIAL_DIR = next(
    (path for path in candidates if (path / "compose.yaml").is_file() and (path / "tutorial_fl_pumba.ipynb").is_file()),
    None,
)
if TUTORIAL_DIR is None:
    raise FileNotFoundError("Abra o notebook da pasta tutorial-fl-pumba ou ajuste TUTORIAL_DIR.")

DATA_DIR = TUTORIAL_DIR / "data"
STATION_FILES = sorted(DATA_DIR.glob("*_Hourly (1).csv"))
MAX_CLIENTS = 6
if len(STATION_FILES) != MAX_CLIENTS:
    raise ValueError("data/ deve conter os seis CSVs de estacoes *_Hourly (1).csv.")
TARGET_COLUMNS = ["PM2.5", "PM10"]
SEQUENCE_LENGTH = 10
FORECAST_HORIZON = 10
PROJECT_NAME = "fl-pumba-tutorial"

### Dados: uma estacao por cliente

Cada CSV representa uma estação, sem divisão entre clientes. O programa utiliza 20 atributos, ignorando a coluna de índice exportada, e prevê os dois poluentes simultaneamente.

Lê o cabeçalho de um CSV para identificar os atributos e monta a configuração comum do modelo. Lista a associação entre clientes e estações, permitindo conferir quais arquivos serão usados.


In [ ]:
FEATURE_COLUMNS = [
    column for column in pd.read_csv(STATION_FILES[0], nrows=0).columns
    if not column.startswith("Unnamed:")
]
config = {
    "features": FEATURE_COLUMNS,
    "targets": TARGET_COLUMNS,
    "sequence_length": SEQUENCE_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
}
for index, path in enumerate(STATION_FILES, start=1):
    print(f"client-{index}: {path.stem}")
print(f"{len(FEATURE_COLUMNS)} atributos; entrada={SEQUENCE_LENGTH}h; previsao={FORECAST_HORIZON}h.")

### Preparacao local

A preparação mantém as observações em cada cliente. Ao escolher menos clientes, o programa apenas deixa de usar as últimas estações da lista, sem redistribuir seus dados.

`prepare_client_data` aceita de 1 a 6 clientes e associa cada identificador a um CSV, em ordem alfabética. Essa seleção representa a distribuição dos dados entre as estações sem criar partições ou cópias.


In [ ]:
def prepare_client_data(num_clients):
    if isinstance(num_clients, bool) or not isinstance(num_clients, int) or not 1 <= num_clients <= MAX_CLIENTS:
        raise ValueError(f"num_clients deve ser um inteiro de 1 a {MAX_CLIENTS}.")
    return {
        f"client-{index + 1}": path
        for index, path in enumerate(STATION_FILES[:num_clients])
    }

print("Estacoes selecionadas por padrao:", [path.stem for path in prepare_client_data(3).values()])

### Cliente: regressao multi-horizonte

O cliente executa a preparação, o treino e a avaliação locais. As próximas células montam seu programa em `client_source`, enviado ao container em memória, sem gerar um script no disco.


#### LSTM e parametros

Define a LSTM de 16 unidades e a saída `[lote, 10 horas, 2 poluentes]`. A variante residual soma uma correção ao último valor observado e começa com correção zero. A persistência apenas repete esse último valor, sem parâmetros treináveis. Cria modelos reproduzíveis e converte seus estados entre PyTorch e NumPy para o protocolo Flower.


In [ ]:
client_source = r'''import argparse
import math
import json
import time
from pathlib import Path

import flwr as fl
import grpc
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(1)


class AirLSTM(nn.Module):
    def __init__(self, features, horizon, targets, target_indices, residual=False):
        super().__init__()
        self.horizon = horizon
        self.targets = targets
        self.target_indices = target_indices
        self.residual = residual
        self.lstm = nn.LSTM(features, 16, batch_first=True)
        self.regressor = nn.Linear(16, horizon * targets)
        if residual:
            nn.init.zeros_(self.regressor.weight)
            nn.init.zeros_(self.regressor.bias)

    def forward(self, sequences):
        output, _ = self.lstm(sequences)
        forecast = self.regressor(output[:, -1, :]).reshape(-1, self.horizon, self.targets)
        if self.residual:
            forecast = forecast + sequences[:, -1, self.target_indices].unsqueeze(1)
        return forecast


class Persistence(nn.Module):
    def __init__(self, target_indices, horizon):
        super().__init__()
        self.register_buffer("target_indices", torch.tensor(target_indices, dtype=torch.long))
        self.horizon = horizon
        self.targets = len(target_indices)

    def forward(self, sequences):
        return sequences[:, -1, self.target_indices].unsqueeze(1).repeat(1, self.horizon, 1)


def create_model(config):
    target_indices = [config["features"].index(target) for target in config["targets"]]
    if config.get("training_mode", "federated") == "persistence":
        return Persistence(target_indices, config["forecast_horizon"])
    with torch.random.fork_rng():
        torch.manual_seed(42)
        return AirLSTM(len(config["features"]), config["forecast_horizon"], len(config["targets"]),
                       target_indices, residual=config.get("model_kind", "standard") == "residual")


def get_parameters(model):
    return [value.detach().cpu().numpy().copy() for value in model.state_dict().values()]


def set_parameters(model, parameters):
    state = model.state_dict()
    if len(parameters) != len(state):
        raise ValueError("Unexpected number of Flower LSTM parameters")
    converted = {}
    for (name, expected), value in zip(state.items(), parameters):
        array = np.asarray(value)
        if array.shape != tuple(expected.shape) or not np.isfinite(array).all():
            raise ValueError(f"Invalid Flower LSTM parameter: {name}")
        converted[name] = torch.tensor(array, dtype=expected.dtype)
    model.load_state_dict(converted, strict=True)


'''

#### Grade horaria, particoes e avaliacao

Carrega os CSVs sem interpolacao e separa cronologicamente **80% para treino e 20% para teste**.
Divide o treino em blocos distintos, um por rodada, sem janelas cruzando fronteiras.
Com cinco rodadas, cada bloco usa aproximadamente 16% do CSV; com quatro, 20%.
A normalizacao usa somente os 80% de treino completos, incluindo blocos posteriores.
O teste permanece fixo. Nao existe particao de validacao.


In [ ]:
client_source += r'''def load_rows(path, config):
    frame = pd.read_csv(path).drop(columns=["Unnamed: 0"], errors="ignore")
    frame.index = pd.to_datetime(frame[["year", "month", "day", "hour"]])
    frame = frame.sort_index()
    if frame.index.has_duplicates:
        raise ValueError(f"{path}: duplicate station timestamps")
    if not (frame.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all():
        raise ValueError(f"{path}: consecutive hourly observations are required")
    if not np.isfinite(frame[config["features"]].to_numpy(dtype=float)).all():
        raise ValueError(f"{path}: station features must be finite")
    train_end = int(len(frame) * 0.8)
    train = frame.iloc[:train_end]
    means = train[config["features"]].mean()
    scales = train[config["features"]].std(ddof=0).replace(0, 1)
    target_means = means[config["targets"]].to_numpy(dtype=np.float64)
    target_scales = scales[config["targets"]].to_numpy(dtype=np.float64)
    length, horizon = config["sequence_length"], config["forecast_horizon"]
    if any(isinstance(value, bool) or not isinstance(value, int) or value < 1 for value in (length, horizon)):
        raise ValueError("Sequence length and forecast horizon must be positive integers")

    def windows(partition):
        features = ((partition[config["features"]] - means) / scales).to_numpy(dtype=np.float32)
        targets = ((partition[config["targets"]] - target_means) / target_scales).to_numpy(dtype=np.float32)
        if len(partition) < length + horizon:
            raise ValueError(f"{path}: each training block and test partition needs {length + horizon} hourly rows")
        return (
            torch.from_numpy(np.stack([features[end-length:end] for end in range(length, len(partition)-horizon+1)])),
            torch.from_numpy(np.stack([targets[end:end+horizon] for end in range(length, len(partition)-horizon+1)])),
        )

    rounds = config["rounds"]
    if isinstance(rounds, bool) or not isinstance(rounds, int) or rounds < 1:
        raise ValueError("Rounds must be a positive integer")
    if train_end // rounds < length + horizon:
        raise ValueError(f"{path}: too many rounds; each training block needs {length + horizon} hourly rows")
    train_blocks = [windows(train.iloc[index * train_end // rounds:(index + 1) * train_end // rounds])
                    for index in range(rounds)]
    test_rows = windows(frame.iloc[train_end:])
    print(f"[data] {Path(path).name}: {len(frame)} hourly records; "
          f"training windows per round={[len(block[0]) for block in train_blocks]}; "
          f"test={len(test_rows[0])}", flush=True)
    return train_blocks, test_rows, target_means, target_scales


def evaluate_rows(model, rows, target_means, target_scales, prediction_path=None):
    model.eval()
    sums = {name: np.zeros(model.targets, dtype=np.float64) for name in ("sse", "sae", "sum_y", "sum_y2")}
    count = 0
    actual_batches, predicted_batches = [], []
    with torch.no_grad():
        for start in range(0, len(rows[0]), 32):
            prediction = model(rows[0][start:start+32]).numpy().astype(np.float64) * target_scales + target_means
            truth = rows[1][start:start+32].numpy().astype(np.float64) * target_scales + target_means
            if prediction_path is not None:
                actual_batches.append(truth.reshape(-1, model.targets))
                predicted_batches.append(prediction.reshape(-1, model.targets))
            error = prediction - truth
            sums["sse"] += np.square(error).sum(axis=(0, 1))
            sums["sae"] += np.abs(error).sum(axis=(0, 1))
            sums["sum_y"] += truth.sum(axis=(0, 1))
            sums["sum_y2"] += np.square(truth).sum(axis=(0, 1))
            count += truth.shape[0] * truth.shape[1]
    if not count or not all(np.isfinite(value).all() for value in sums.values()):
        raise ValueError("Invalid regression evaluation statistics")
    if prediction_path is not None:
        prediction_path.write_text(json.dumps({
            "actual": np.concatenate(actual_batches).tolist(),
            "predicted": np.concatenate(predicted_batches).tolist(),
        }, allow_nan=False), encoding="utf-8")
    stats = {name: value.tolist() for name, value in sums.items()}
    stats["count"] = count
    return float(sums["sse"].mean() / count), len(rows[0]), {
        "regression_stats": json.dumps(stats),
    }


'''

#### Treinamento no cliente Flower

Executa ate `epochs` epocas no bloco atual. Por padrao, `early_stopping=True` monitora
a media do MSE normalizado dos minibatches de treino, ponderada pelo numero de janelas.
Com `patience=2`, para apos duas epocas consecutivas sem progresso de pelo menos
`min_delta=0.0001` desde a ultima melhora significativa. Com delta zero, exige queda estrita.
Restaura os pesos da epoca com menor perda observada, mesmo se atingir o limite.
Contadores e checkpoints sao reiniciados a cada rodada. Sem early stopping, executa
todas as epocas e retorna os ultimos pesos. O teste nunca e acessado durante `fit`.
Essa perda e medida durante as atualizacoes, nao em uma reavaliacao ao fim da epoca.
Parar por perda de treino economiza trabalho, mas nao detecta sobreajuste de forma confiavel.
O modo federado recebe pesos globais; o local conserva seus pesos; persistencia nao treina.
Os resultados registram epocas executadas, epoca retornada e atualizacoes por cliente/rodada.


In [ ]:
client_source += r'''class AirClient(fl.client.NumPyClient):
    def __init__(self, client_id, config, train_blocks, test_rows, target_means, target_scales):
        self.client_id = client_id
        self.training_mode = config.get("training_mode", "federated")
        self.train_blocks = train_blocks
        self.epochs = config["epochs"]
        self.learning_rate = config["learning_rate"]
        if (isinstance(self.learning_rate, bool) or not isinstance(self.learning_rate, (int, float))
                or not math.isfinite(self.learning_rate) or self.learning_rate <= 0):
            raise ValueError("Learning rate must be a finite positive number")
        if isinstance(self.epochs, bool) or not isinstance(self.epochs, int) or self.epochs < 1:
            raise ValueError("Epochs must be a positive integer")
        self.early_stopping = config["early_stopping"]
        self.patience = config["patience"]
        self.min_delta = config["min_delta"]
        if not isinstance(self.early_stopping, bool):
            raise ValueError("Early stopping must be a boolean")
        if isinstance(self.patience, bool) or not isinstance(self.patience, int) or self.patience < 1:
            raise ValueError("Patience must be a positive integer")
        if (isinstance(self.min_delta, bool) or not isinstance(self.min_delta, (int, float))
                or not math.isfinite(self.min_delta) or self.min_delta < 0):
            raise ValueError("Min delta must be a finite non-negative number")
        self.collect_predictions = config.get("collect_predictions", True)
        if not isinstance(self.collect_predictions, bool):
            raise ValueError("Collect predictions must be a boolean")
        self.total_rounds = config["rounds"]
        self.test_rows = test_rows
        self.target_means, self.target_scales = target_means, target_scales
        self.model = create_model(config)

    def get_parameters(self, config):
        return get_parameters(self.model)

    def fit(self, parameters, config):
        round_number = int(config["server_round"])
        if self.training_mode == "federated" or round_number == 1:
            set_parameters(self.model, parameters)
        if not 1 <= round_number <= len(self.train_blocks):
            raise ValueError(f"{self.client_id}: round {round_number} has no training block")
        train_rows = self.train_blocks[round_number - 1]
        if self.training_mode == "persistence":
            return self.get_parameters(config), len(train_rows[0]), {
                "client_id": self.client_id, "server_round": round_number,
                "epochs_completed": 0, "optimizer_steps": 0,
                "early_stopped": False, "selected_epoch": 0,
            }
        generator = torch.Generator().manual_seed(round_number - 1 + sum(map(ord, self.client_id)))
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        optimizer_steps = 0
        best_loss = reference_loss = math.inf
        best_parameters = None
        best_epoch = epochs_completed = epochs_without_improvement = 0
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            total_loss = 0.0
            order = torch.randperm(len(train_rows[0]), generator=generator)
            for indices in order.split(32):
                features, targets = train_rows[0][indices], train_rows[1][indices]
                optimizer.zero_grad()
                loss = nn.functional.mse_loss(self.model(features), targets)
                if not torch.isfinite(loss):
                    raise ValueError(f"{self.client_id}: non-finite training loss")
                loss.backward()
                optimizer.step()
                optimizer_steps += 1
                total_loss += loss.item() * len(indices)
            epoch_loss = total_loss / len(train_rows[0])
            epochs_completed = epoch
            if self.early_stopping:
                if epoch_loss < best_loss:
                    best_loss, best_epoch = epoch_loss, epoch
                    best_parameters = self.get_parameters(config)
                if epoch_loss < reference_loss and reference_loss - epoch_loss >= self.min_delta:
                    reference_loss = epoch_loss
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1
                if epochs_without_improvement >= self.patience:
                    break
        selected_epoch = epochs_completed
        if self.early_stopping:
            if best_parameters is None:
                raise RuntimeError(f"{self.client_id}: no finite training checkpoint")
            set_parameters(self.model, best_parameters)
            selected_epoch = best_epoch
        early_stopped = epochs_completed < self.epochs
        print(f"[{self.client_id}] fit round {round_number}; epochs={epochs_completed}/{self.epochs}; "
              f"selected_epoch={selected_epoch}; early_stopped={early_stopped}; "
              f"training_windows={len(train_rows[0])}; last_train_mse={epoch_loss:.6f}; "
              f"learning_rate={self.learning_rate:g}; optimizer_steps={optimizer_steps}", flush=True)
        return self.get_parameters(config), len(train_rows[0]), {
            "client_id": self.client_id, "server_round": round_number,
            "epochs_completed": epochs_completed, "optimizer_steps": optimizer_steps,
            "early_stopped": early_stopped, "selected_epoch": selected_epoch,
        }

    def evaluate(self, parameters, config):
        if self.training_mode == "federated":
            set_parameters(self.model, parameters)
        prediction_path = None
        if self.collect_predictions and config["server_round"] == self.total_rounds:
            prediction_path = Path("/predictions/final.json")
        loss, count, metrics = evaluate_rows(
            self.model, self.test_rows, self.target_means, self.target_scales, prediction_path,
        )
        return loss, count, {**metrics, "client_id": self.client_id}


'''

#### Inicializacao do cliente

Lê os argumentos enviados ao container, prepara os dados da estação e conecta o cliente ao servidor Flower. A conexão sem TLS é uma simplificação da rede Docker deste programa.


In [ ]:
client_source += r'''def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--id", required=True)
    parser.add_argument("--data", required=True)
    parser.add_argument("--config", required=True, help="Model configuration as JSON, not a file")
    parser.add_argument("--host", default="server")
    parser.add_argument("--port", type=int, default=8080)
    args = parser.parse_args()
    config = json.loads(args.config)
    client = AirClient(args.id, config, *load_rows(args.data, config))
    # When a round deadline expires, the server cancels this client's stream.
    # Reconnecting lets a late client rejoin later rounds instead of leaving for good.
    for attempt in range(1, 101):
        try:
            fl.client.start_client(
                server_address=f"{args.host}:{args.port}", client=client.to_client(),
                insecure=True, max_wait_time=300 if attempt == 1 else 60,
            )
            return
        except grpc.RpcError as error:
            print(f"[{args.id}] connection lost ({error.code().name}); reconnecting (attempt {attempt})", flush=True)
            time.sleep(1)
    raise RuntimeError(f"{args.id}: too many reconnections")


if __name__ == "__main__":
    main()
'''

### Servidor

O servidor coordena as rodadas e agrega parâmetros, sem carregar os CSVs das estações. As próximas células montam seu programa em `server_source`. A localidade dos dados não oferece, por si só, privacidade formal.


#### Metricas de regressao

Combina as somas de erros e de alvos recebidas dos clientes para calcular RMSE, MAE e R² globais e por poluente. Isso evita fazer média de R² locais; a célula também verifica os limites de participação aceitos pelo servidor.


In [ ]:
server_source = r'''import argparse
import json
import time
from pathlib import Path

import flwr as fl
import numpy as np

from client import create_model, get_parameters


def regression_metrics(metrics):
    if not metrics:
        raise ValueError("No regression evaluation metrics received")
    combined = {key: np.zeros(2, dtype=np.float64) for key in ("sse", "sae", "sum_y", "sum_y2")}
    count = 0
    for num_examples, values in metrics:
        stats = json.loads(values["regression_stats"])
        if num_examples <= 0 or stats["count"] <= 0:
            raise ValueError("Invalid regression evaluation count")
        for key in combined:
            array = np.asarray(stats[key], dtype=np.float64)
            if array.shape != (2,) or not np.isfinite(array).all():
                raise ValueError(f"Invalid regression statistic: {key}")
            if key in ("sse", "sae", "sum_y2") and np.any(array < 0):
                raise ValueError(f"Negative regression statistic: {key}")
            combined[key] += array
        count += stats["count"]
    mse = combined["sse"] / count
    mae = combined["sae"] / count
    total_variation = np.maximum(combined["sum_y2"] - combined["sum_y"] ** 2 / count, 0)
    r2 = np.ones(2, dtype=np.float64)
    np.divide(combined["sse"], total_variation, out=r2, where=total_variation > 0)
    r2 = np.where(total_variation > 0, 1 - r2, np.where(combined["sse"] == 0, 1.0, 0.0))
    result = {"rmse": float(np.sqrt(mse.mean())), "mae": float(mae.mean()), "r2": float(r2.mean())}
    for index, target in enumerate(("pm25", "pm10")):
        result.update({
            f"rmse_{target}": float(np.sqrt(mse[index])),
            f"mae_{target}": float(mae[index]), f"r2_{target}": float(r2[index]),
        })
    return result


def validate_participation(clients, min_fit_clients, round_timeout):
    for name, value in (("clients", clients), ("min_fit_clients", min_fit_clients), ("round_timeout", round_timeout)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f"{name} must be a positive integer")
    if not min_fit_clients <= clients <= 6:
        raise ValueError("Require 1 <= min_fit_clients <= clients <= 6")
'''

#### Avaliacao distribuida e participacao parcial

A avaliacao usa os 20% de teste de cada cliente, sem enviar observacoes individuais.
No modo completo, todos respondem; no parcial, as metricas cobrem apenas quem respondeu.
O servidor calcula resultados globais e por estacao/poluente a partir dessas estatisticas.


#### Agregacao FedAvg

`TimedFedAvg` estende a estratégia do Flower para aguardar conexões e a liberação do Pumba, medir tempos e conferir a participação. A agregação mantém o FedAvg, ponderando os parâmetros pelo número de janelas de treino.


In [ ]:
server_source += r'''class TimedFedAvg(fl.server.strategy.FedAvg):
    def __init__(self, clients, startup_delay, start_signal=None,
                 allow_partial=True, training_mode="federated", **kwargs):
        super().__init__(**kwargs)
        self.clients = clients
        self.startup_delay = startup_delay
        self.start_signal = start_signal
        self.round_started = None
        self.train_examples = 0
        self.allow_partial = allow_partial
        self.training_mode = training_mode
        self.training_work = {}
        self.training_started = None
        self.fit_participants = []
        self.fit_failures = 0
        self.fit_seconds = 0.0

    def configure_fit(self, server_round, parameters, client_manager):
        if server_round == 1:
            if not client_manager.wait_for(self.clients, timeout=300):
                raise TimeoutError("Not all clients connected before training")
            print("[server] All clients connected", flush=True)
            if self.start_signal:
                deadline = time.monotonic() + 300
                print("[server] Waiting for verified Pumba rules", flush=True)
                while not Path(self.start_signal).is_file():
                    if time.monotonic() >= deadline:
                        raise TimeoutError("Pumba verification did not release training")
                    time.sleep(0.2)
            print(f"[server] Waiting {self.startup_delay}s for optional Pumba startup", flush=True)
            time.sleep(self.startup_delay)
        self.round_started = time.monotonic()
        if self.training_started is None:
            self.training_started = self.round_started
        instructions = super().configure_fit(server_round, parameters, client_manager)
        if self.allow_partial:
            self.min_available_clients = self.min_fit_clients
        return instructions

    def aggregate_fit(self, server_round, results, failures):
        if not self.allow_partial and (failures or len(results) != self.clients):
            raise RuntimeError(f"Round {server_round}: incomplete Flower fit ({len(results)} clients, {len(failures)} failures)")
        if len(results) < self.min_fit_clients:
            raise RuntimeError(f"Round {server_round}: only {len(results)} fit results; minimum is {self.min_fit_clients}")
        participants = []
        for _, result in results:
            client_id = result.metrics.get("client_id")
            if (client_id not in {f"client-{index}" for index in range(1, self.clients + 1)}
                    or result.metrics.get("server_round") != server_round
                    or result.num_examples <= 0):
                raise RuntimeError(f"Round {server_round}: invalid client identity, round or example count")
            participants.append(client_id)
        if len(set(participants)) != len(participants):
            raise RuntimeError(f"Round {server_round}: duplicate client identities")
        self.fit_participants = sorted(participants)
        self.fit_failures = len(failures)
        self.fit_seconds = time.monotonic() - self.round_started
        self.train_examples = sum(result.num_examples for _, result in results)
        self.training_work = {result.metrics["client_id"]: {
            "windows": result.num_examples, "epochs": result.metrics["epochs_completed"],
            "optimizer_steps": result.metrics["optimizer_steps"],
            "early_stopped": result.metrics["early_stopped"],
            "selected_epoch": result.metrics["selected_epoch"],
        } for _, result in results}
        if self.training_mode != "federated":
            return results[0][1].parameters, {}
        return super().aggregate_fit(server_round, results, failures)

'''

#### Registro das rodadas

Registra métricas, tempos e participantes de cada rodada em `ROUND_RESULT`, para o notebook recuperar os resultados dos logs. Verifica as respostas de avaliação e identifica as estações avaliadas, especialmente importantes no modo parcial.


In [ ]:
server_source += r'''    def record_round(self, server_round, loss, metrics, evaluation_participants,
                                         station_metrics):
        result = {
            "round": server_round,
            "duration_seconds": round(time.monotonic() - self.round_started, 3),
            "cumulative_seconds": round(time.monotonic() - self.training_started, 3),
            "fit_seconds": round(self.fit_seconds, 3),
            "test_loss": float(loss),
            "training_work": self.training_work,
            "training_mode": self.training_mode,
            "station_metrics": station_metrics,
            **{f"test_{key}": float(value) for key, value in metrics.items()},
            "clients": len(self.fit_participants),
            "fit_participants": self.fit_participants,
            "missing_clients": sorted(
                {f"client-{index}" for index in range(1, self.clients + 1)} - set(self.fit_participants)
            ),
            "fit_failures": self.fit_failures,
            "evaluation_clients": len(evaluation_participants),
            "evaluation_participants": evaluation_participants,
            "examples": self.train_examples,
        }
        print("ROUND_RESULT " + json.dumps(result), flush=True)

    def aggregate_evaluate(self, server_round, results, failures):
        if (not self.allow_partial and (failures or len(results) != self.clients)) or len(results) < self.min_evaluate_clients:
            raise RuntimeError(f"Round {server_round}: incomplete Flower evaluation ({len(results)} clients, {len(failures)} failures)")
        participants = sorted(result.metrics["client_id"] for _, result in results)
        expected = {f"client-{index}" for index in range(1, self.clients + 1)}
        if len(set(participants)) != len(participants) or not set(participants) <= expected:
            raise ValueError("Invalid or duplicate evaluation client identities")
        loss, metrics = super().aggregate_evaluate(server_round, results, failures)
        station_metrics = {result.metrics["client_id"]: {
            "test": regression_metrics([(result.num_examples, result.metrics)]),
        } for _, result in results}
        self.record_round(server_round, loss, metrics, participants, station_metrics)
        return loss, metrics


'''

#### Inicializacao do servidor

Lê a configuração do servidor, inicializa os parâmetros e configura FedAvg, limites de participação e agregação das métricas. Inicia o serviço que solicita treinamento e avaliação distribuídos durante as rodadas.


In [ ]:
server_source += r'''def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--rounds", type=int, default=5)
    parser.add_argument("--clients", type=int, default=3)
    parser.add_argument("--startup-delay", type=int, default=15)
    parser.add_argument("--start-signal")
    parser.add_argument("--port", type=int, default=8080)
    parser.add_argument("--allow-partial", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--min-fit-clients", type=int)
    parser.add_argument("--round-timeout", type=int, default=20)
    args = parser.parse_args()
    minimum = (1 if args.allow_partial else args.clients) if args.min_fit_clients is None else args.min_fit_clients
    validate_participation(args.clients, minimum, args.round_timeout)
    if not args.allow_partial and minimum != args.clients:
        parser.error("--min-fit-clients below --clients requires --allow-partial")

    config = json.loads(args.config)
    parameters = get_parameters(create_model(config))
    strategy = TimedFedAvg(
        clients=args.clients, startup_delay=args.startup_delay,
        start_signal=args.start_signal,
        allow_partial=args.allow_partial,
        training_mode=config.get("training_mode", "federated"),
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=minimum, min_evaluate_clients=minimum,
        min_available_clients=args.clients, accept_failures=args.allow_partial,
        initial_parameters=fl.common.ndarrays_to_parameters(parameters),
        on_fit_config_fn=lambda server_round: {"server_round": server_round},
        on_evaluate_config_fn=lambda server_round: {"server_round": server_round},
        evaluate_metrics_aggregation_fn=regression_metrics,
    )
    completed = False
    try:
        fl.server.start_server(
            server_address=f"0.0.0.0:{args.port}",
            config=fl.server.ServerConfig(num_rounds=args.rounds, round_timeout=args.round_timeout),
            strategy=strategy,
        )
        completed = True
    finally:
        if not completed:
            print("[server] Training failed", flush=True)

    print("[server] Training complete", flush=True)


if __name__ == "__main__":
    main()
'''

Verifica a sintaxe dos programas e prepara um módulo de cliente em memória para o servidor reutilizar a definição do modelo. Assim, ambos podem executar com `python -c`, sem criar arquivos Python.


In [ ]:
import ast

ast.parse(client_source)
ast.parse(server_source)
server_runtime_source = (
    "import sys, types\n"
    "client = types.ModuleType('client')\n"
    "sys.modules['client'] = client\n"
    f"exec(compile({client_source!r}, '<embedded-client>', 'exec'), client.__dict__)\n"
    + server_source
)

### Orquestração dos experimentos

A orquestração conecta o notebook ao Docker: monta comandos, inicia os serviços e acompanha os experimentos. A célula abaixo constrói o comando Compose e pode habilitar o perfil de caos e a configuração recebida pela entrada padrão.

`compose_command` monta a chamada ao Docker Compose com o nome exclusivo do projeto. Os argumentos opcionais ativam o perfil de caos e a leitura do manifesto dinâmico pela entrada padrão.


In [ ]:
def compose_command(profile=None, include_stdin_config=False):
    command = ["docker", "compose", "--project-name", PROJECT_NAME]
    if profile:
        command.extend(["--profile", profile])
    command.extend(["-f", "compose.yaml"])
    if include_stdin_config:
        command.extend(["-f", "-"])
    return command

`run_docker_command` executa comandos Docker, captura suas saídas e informa falhas. Também encaminha o Compose dinâmico em memória, evitando gravar um arquivo de configuração adicional.


In [ ]:
def run_docker_command(command, failure_message, show_output=True, compose_config=None):
    completed = subprocess.run(
        command, cwd=TUTORIAL_DIR, capture_output=True, text=True,
        input=compose_config,
    )
    if show_output and completed.stdout:
        print(completed.stdout, end="")
    if show_output and completed.stderr:
        print(completed.stderr, end="")
    if completed.returncode != 0:
        raise RuntimeError(
            f"{failure_message}\nComando: {subprocess.list2cmdline(command)}\n"
            f"Exit code: {completed.returncode}\n{completed.stderr or completed.stdout}"
        )
    return completed


Importa o suporte a expressões regulares e define as imagens do Pumba e da ferramenta `tc`. Elas permitem aplicar e conferir as interferências de rede nos containers dos clientes.


In [ ]:
import re

PUMBA_IMAGE = "gaiaadm/pumba:latest"
TC_IMAGE = "ghcr.io/alexei-led/pumba-alpine-nettools:latest"

#### Configuração do cenário

A configuração do cenário define estações participantes, parâmetros do treinamento e interferências de rede antes de iniciar os containers.


#### Validacao dos parametros

Valida os argumentos do experimento, incluindo quantidade de estações, rodadas, tempos e parâmetros de rede. Define a participação mínima e os clientes afetados, evitando iniciar um cenário inconsistente.


In [ ]:
def validate_scenario_parameters(num_clients, rounds, chaos, timeout_seconds, delay_time, delay_jitter, loss_percent, delay_distribution, interface, duration, allow_partial, min_fit_clients, round_timeout_seconds, chaos_clients):
    for label, value in (("num_clients", num_clients), ("rounds", rounds),
                         ("timeout_seconds", timeout_seconds)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f"{label} deve ser um inteiro positivo.")
    prepare_client_data(num_clients)
    for label, value in (("delay_time", delay_time), ("delay_jitter", delay_jitter)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 0:
            raise ValueError(f"{label} deve ser um inteiro nao negativo.")
    if (isinstance(loss_percent, bool) or not isinstance(loss_percent, (int, float))
            or not math.isfinite(loss_percent) or not 0 <= loss_percent <= 100):
        raise ValueError("loss_percent deve estar entre 0 e 100.")
    if delay_distribution not in ("normal", "uniform", "pareto", "paretonormal"):
        raise ValueError("delay_distribution deve ser normal, uniform, pareto ou paretonormal.")
    if not isinstance(interface, str) or not re.fullmatch(r"[A-Za-z0-9_.-]{1,15}", interface):
        raise ValueError("interface deve ser um nome de interface Linux valido.")
    if not isinstance(chaos, bool):
        raise ValueError("chaos deve ser True ou False.")
    if chaos and (delay_jitter and not delay_time):
        raise ValueError("delay_jitter exige delay_time maior que zero.")
    duration = timeout_seconds + 60 if duration is None else duration
    if isinstance(duration, bool) or not isinstance(duration, int) or duration < timeout_seconds:
        raise ValueError("duration deve ser inteiro e cobrir timeout_seconds.")
    client_ids = tuple(f"client-{index + 1}" for index in range(num_clients))
    if not isinstance(allow_partial, bool):
        raise ValueError("allow_partial deve ser True ou False.")
    minimum = (1 if allow_partial else num_clients) if min_fit_clients is None else min_fit_clients
    for label, value in (("min_fit_clients", minimum), ("round_timeout_seconds", round_timeout_seconds)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f"{label} deve ser um inteiro positivo.")
    if minimum > num_clients or (not allow_partial and minimum != num_clients):
        raise ValueError("min_fit_clients deve ser <= num_clients; valor menor exige allow_partial=True.")
    if chaos_clients is not None and (not isinstance(chaos_clients, (tuple, list))
            or any(not isinstance(value, str) for value in chaos_clients)):
        raise ValueError("chaos_clients deve ser uma lista/tupla de nomes de clientes.")
    targets = client_ids if chaos_clients is None else tuple(chaos_clients)
    if (not targets or len(set(targets)) != len(targets)
            or any(client_id not in client_ids for client_id in targets)):
        raise ValueError("chaos_clients deve conter clientes existentes, sem repeticoes.")
    return duration, client_ids, minimum, targets

def validate_pumba_effects(chaos, delay_time, delay_jitter, delay_correlation,
                           loss_percent, loss_correlation, duplicate_percent,
                           duplicate_correlation, corrupt_percent, corrupt_correlation,
                           rate_value, rate_packet_overhead, rate_cell_size, rate_cell_overhead):
    if not isinstance(chaos, bool):
        raise ValueError("chaos deve ser True ou False.")
    for name, value in (("delay_time", delay_time), ("delay_jitter", delay_jitter),
                        ("rate_packet_overhead", rate_packet_overhead),
                        ("rate_cell_size", rate_cell_size), ("rate_cell_overhead", rate_cell_overhead)):
        if isinstance(value, bool) or not isinstance(value, int) or value < 0:
            raise ValueError(f"{name} deve ser um inteiro nao negativo.")
    for name, value in (("delay_correlation", delay_correlation),
                        ("loss_percent", loss_percent), ("loss_correlation", loss_correlation),
                        ("duplicate_percent", duplicate_percent),
                        ("duplicate_correlation", duplicate_correlation),
                        ("corrupt_percent", corrupt_percent),
                        ("corrupt_correlation", corrupt_correlation)):
        if (isinstance(value, bool) or not isinstance(value, (int, float))
                or not math.isfinite(value) or not 0 <= value <= 100):
            raise ValueError(f"{name} deve estar entre 0 e 100.")
    if delay_jitter and not delay_time:
        raise ValueError("delay_jitter exige delay_time maior que zero.")
    if rate_value is not None and (
            not isinstance(rate_value, str)
            or not re.fullmatch(r"[0-9]+(?:\.[0-9]+)?(?:bit|kbit|mbit|gbit|tbit)", rate_value, re.IGNORECASE)):
        raise ValueError("rate_value deve usar uma unidade Pumba, por exemplo '100kbit' ou '1mbit'.")
    effects = {
        "delay": delay_time > 0,
        "loss": loss_percent > 0,
        "duplicate": duplicate_percent > 0,
        "corrupt": corrupt_percent > 0,
        "rate": rate_value is not None,
    }
    correlations = {
        "delay_correlation": (effects["delay"], delay_correlation),
        "loss_correlation": (effects["loss"], loss_correlation),
        "duplicate_correlation": (effects["duplicate"], duplicate_correlation),
        "corrupt_correlation": (effects["corrupt"], corrupt_correlation),
    }
    for name, (enabled, value) in correlations.items():
        if value and not enabled:
            raise ValueError(f"{name} exige que seu efeito correspondente esteja ativo.")
    if chaos and not any(effects.values()):
        raise ValueError("chaos=True exige ao menos um efeito de rede ativo.")
    configured_nondefaults = (
        delay_time != 0 or delay_jitter != 0 or delay_correlation != 0
        or loss_percent != 0 or loss_correlation != 0
        or duplicate_percent != 0 or duplicate_correlation != 0
        or corrupt_percent != 0 or corrupt_correlation != 0 or rate_value is not None
        or rate_packet_overhead != 0 or rate_cell_size != 0 or rate_cell_overhead != 0
    )
    if not chaos and configured_nondefaults:
        raise ValueError("Ative chaos=True para aplicar os parametros de caos informados.")
    return effects


Monta o manifesto Compose em memória com os comandos de servidor, clientes e Pumba opcional. Cada cliente recebe somente o CSV da sua estação em leitura, preservando a separação dos dados durante a simulação.


In [ ]:
def scenario_configuration(num_clients, rounds, chaos, timeout_seconds, delay_time,
                           delay_jitter, loss_percent, delay_distribution, interface, duration,
                           allow_partial=True, min_fit_clients=None,
                           round_timeout_seconds=20, chaos_clients=None, epochs=50, learning_rate=0.001,
                           training_mode="federated", model_kind="standard",
                           early_stopping=True, patience=2, min_delta=0.0001, collect_predictions=True,
                           delay_correlation=0, loss_correlation=0,
                           duplicate_percent=0, duplicate_correlation=0,
                           corrupt_percent=0, corrupt_correlation=0, rate_value=None,
                           rate_packet_overhead=0, rate_cell_size=0, rate_cell_overhead=0):
    effects = validate_pumba_effects(
        chaos, delay_time, delay_jitter, delay_correlation, loss_percent, loss_correlation,
        duplicate_percent, duplicate_correlation, corrupt_percent, corrupt_correlation,
        rate_value, rate_packet_overhead, rate_cell_size, rate_cell_overhead,
    )
    if not isinstance(collect_predictions, bool):
        raise ValueError("collect_predictions deve ser True ou False.")
    if training_mode not in ("federated", "local", "persistence"):
        raise ValueError("training_mode deve ser federated, local ou persistence.")
    if model_kind not in ("standard", "residual"):
        raise ValueError("model_kind deve ser standard ou residual.")
    if training_mode == "persistence" and model_kind != "standard":
        raise ValueError("Persistencia nao utiliza um modelo residual.")
    if (isinstance(learning_rate, bool) or not isinstance(learning_rate, (int, float))
            or not math.isfinite(learning_rate) or learning_rate <= 0):
        raise ValueError("learning_rate deve ser um numero positivo e finito.")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or epochs < 1:
        raise ValueError("epochs deve ser um inteiro positivo.")
    if not isinstance(early_stopping, bool):
        raise ValueError("early_stopping deve ser True ou False.")
    if isinstance(patience, bool) or not isinstance(patience, int) or patience < 1:
        raise ValueError("patience deve ser um inteiro positivo.")
    if (isinstance(min_delta, bool) or not isinstance(min_delta, (int, float))
            or not math.isfinite(min_delta) or min_delta < 0):
        raise ValueError("min_delta deve ser um numero nao negativo e finito.")
    duration, client_ids, minimum, targets = validate_scenario_parameters(
        num_clients,
        rounds,
        chaos,
        timeout_seconds,
        delay_time,
        delay_jitter,
        loss_percent,
        delay_distribution,
        interface,
        duration,
        allow_partial,
        min_fit_clients,
        round_timeout_seconds,
        chaos_clients
    )
    round_config = {**config, "rounds": rounds, "epochs": epochs, "learning_rate": learning_rate,
                    "training_mode": training_mode, "model_kind": model_kind,
                    "early_stopping": early_stopping, "patience": patience, "min_delta": min_delta,
                    "collect_predictions": collect_predictions}
    server_command = ["python", "-c", server_runtime_source, "--config", json.dumps(round_config),
                      "--rounds", str(rounds), "--clients", str(num_clients), "--startup-delay", "0"]
    server_command.extend(["--round-timeout", str(round_timeout_seconds),
                           "--min-fit-clients", str(minimum)])
    server_command.append("--allow-partial" if allow_partial else "--no-allow-partial")
    if chaos:
        server_command.extend(["--start-signal", "/data/start-training.flag"])
    services = {"server": {
        "command": server_command,
        "healthcheck": {
            "test": ["CMD", "python", "-c",
                     "import socket; socket.create_connection(('127.0.0.1', 8080), timeout=1).close()"],
            "interval": "2s", "timeout": "2s", "retries": 150,
        },
    }}
    stations = prepare_client_data(num_clients)
    for client_id in client_ids:
        services[client_id] = {
            "image": "fl-pumba-tutorial-flower:1.33.0",
            "container_name": f"{PROJECT_NAME}-{client_id}",
            "command": ["python", "-c", client_source, "--id", client_id,
                        "--data", "/station.csv", "--config", json.dumps(round_config)],
            "volumes": [{"type": "bind", "source": str(stations[client_id].resolve()),
                         "target": "/station.csv", "read_only": True}],
            "depends_on": {"server": {"condition": "service_healthy"}}, "networks": ["workshop"],
        }
    pumba = None
    if chaos:
        command = ["netem", "--duration", f"{duration}s", "--interface", interface, "--tc-image", TC_IMAGE]
        active_effects = [name for name, active in effects.items() if active]
        if len(active_effects) > 1:
            command.append("combine")
            for effect in active_effects:
                command.append(f"--{effect}")
                if effect == "delay":
                    command.extend(["--delay-time", str(delay_time),
                                    "--delay-jitter", str(delay_jitter),
                                    "--delay-correlation", str(delay_correlation)])
                    if delay_jitter:
                        command.extend(["--delay-distribution", delay_distribution])
                elif effect == "loss":
                    command.extend(["--loss-percent", str(loss_percent),
                                    "--loss-correlation", str(loss_correlation)])
                elif effect == "duplicate":
                    command.extend(["--duplicate-percent", str(duplicate_percent),
                                    "--duplicate-correlation", str(duplicate_correlation)])
                elif effect == "corrupt":
                    command.extend(["--corrupt-percent", str(corrupt_percent),
                                    "--corrupt-correlation", str(corrupt_correlation)])
                else:
                    command.extend(["--rate-value", rate_value,
                                    "--rate-packet-overhead", str(rate_packet_overhead),
                                    "--rate-cell-size", str(rate_cell_size),
                                    "--rate-cell-overhead", str(rate_cell_overhead)])
            command.extend(["--"] + [f"{PROJECT_NAME}-{client_id}" for client_id in targets])
        else:
            effect = active_effects[0]
            command.append(effect)
            if effect == "delay":
                command.extend(["--time", str(delay_time), "--jitter", str(delay_jitter),
                                "--correlation", str(delay_correlation)])
                if delay_jitter:
                    command.extend(["--distribution", delay_distribution])
            elif effect == "loss":
                command.extend(["--percent", str(loss_percent),
                                "--correlation", str(loss_correlation)])
            elif effect == "duplicate":
                command.extend(["--percent", str(duplicate_percent),
                                "--correlation", str(duplicate_correlation)])
            elif effect == "corrupt":
                command.extend(["--percent", str(corrupt_percent),
                                "--correlation", str(corrupt_correlation)])
            else:
                command.extend(["--rate", rate_value,
                                "--packetoverhead", str(rate_packet_overhead),
                                "--cellsize", str(rate_cell_size),
                                "--celloverhead", str(rate_cell_overhead)])
            command.extend([f"{PROJECT_NAME}-{client_id}" for client_id in targets])
        services["pumba"] = {
            "image": PUMBA_IMAGE, "container_name": f"{PROJECT_NAME}-pumba",
            "profiles": ["chaos"], "command": command,
            "volumes": ["/var/run/docker.sock:/var/run/docker.sock:ro"],
            "depends_on": list(client_ids), "networks": ["workshop"],
        }
        pumba = {"delay_time": delay_time, "delay_jitter": delay_jitter,
                 "delay_correlation": delay_correlation, "loss_percent": loss_percent,
                 "loss_correlation": loss_correlation,
                 "duplicate_percent": duplicate_percent,
                 "duplicate_correlation": duplicate_correlation,
                 "corrupt_percent": corrupt_percent,
                 "corrupt_correlation": corrupt_correlation,
                 "rate_value": rate_value, "rate_packet_overhead": rate_packet_overhead,
                 "rate_cell_size": rate_cell_size, "rate_cell_overhead": rate_cell_overhead,
                 "delay_distribution": delay_distribution if delay_time and delay_jitter else None,
                 "interface": interface, "duration": duration, "clients": list(targets)}
    return {"services": services}, client_ids, pumba

#### Verificação do Pumba

Para atribuir os efeitos observados ao caos solicitado, o programa verifica se o Pumba está ativo e se as regras de rede foram realmente aplicadas.

`netem_matches` interpreta a saída do `tc`, converte as unidades de tempo e compara atraso, jitter e perda com a configuração. A verificação confirma se a interferência medida corresponde à solicitada.


In [ ]:
def netem_matches(output, settings):
    if "qdisc netem" not in output:
        return False
    if settings["delay_time"]:
        match = re.search(r"\bdelay ([\d.]+)(us|ms|s)(?:\s+([\d.]+)(us|ms|s))?", output)
        if not match:
            return False
        units = {"us": 0.001, "ms": 1, "s": 1000}
        delay = float(match[1]) * units[match[2]]
        jitter = float(match[3]) * units[match[4]] if match[3] else 0
        if not math.isclose(delay, settings["delay_time"]) or not math.isclose(jitter, settings["delay_jitter"]):
            return False
    for effect, key in (("loss", "loss_percent"),
                        ("duplicate", "duplicate_percent"),
                        ("corrupt", "corrupt_percent")):
        match = re.search(rf"\b{effect} ([\d.]+)%", output)
        actual = float(match[1]) if match else 0
        if not math.isclose(actual, settings[key], abs_tol=0.001):
            return False
    if settings["rate_value"]:
        rate = re.search(r"\brate ([\d.]+)([kmgt]?bit)", output, re.IGNORECASE)
        if not rate:
            return False
        units = {"bit": 1, "kbit": 1e3, "mbit": 1e6, "gbit": 1e9, "tbit": 1e12}
        actual = float(rate[1]) * units[rate[2].lower()]
        expected_match = re.fullmatch(r"([\d.]+)([kmgt]?bit)", settings["rate_value"], re.IGNORECASE)
        if not expected_match or not math.isclose(
                actual, float(expected_match[1]) * units[expected_match[2].lower()], rel_tol=0.01):
            return False
    return True


`require_pumba_running` consulta o estado do container Pumba e interrompe o experimento com logs se ele não estiver ativo. Evita tratar uma execução sem caos como se as regras ainda estivessem sendo aplicadas.


In [ ]:
def require_pumba_running():
    status = run_docker_command(
        ["docker", "inspect", "--format", "{{.State.Status}}", f"{PROJECT_NAME}-pumba"],
        "Nao foi possivel verificar o container Pumba.", show_output=False,
    ).stdout.strip()
    if status != "running":
        logs = run_docker_command(["docker", "logs", f"{PROJECT_NAME}-pumba"], "Falha ao ler logs Pumba.")
        raise RuntimeError(f"Pumba nao esta ativo ({status}). {logs.stdout}{logs.stderr}")


`verify_pumba` consulta as regras `tc` de cada cliente afetado até confirmá-las ou atingir o prazo. O treino só é liberado após essa confirmação, mantendo o início das rodadas condicionado ao caos configurado.


In [ ]:
def verify_pumba(client_ids, settings, deadline):
    verified = {}
    last_output = {}
    while time.monotonic() < deadline:
        require_pumba_running()
        for client_id in client_ids:
            if client_id in verified:
                continue
            output = run_docker_command(
                ["docker", "run", "--rm", "--network", f"container:{PROJECT_NAME}-{client_id}",
                 "--entrypoint", "tc", TC_IMAGE, "-s", "qdisc", "show", "dev", settings["interface"]],
                f"Falha ao verificar tc em {client_id}.", show_output=False,
            ).stdout
            last_output[client_id] = output
            if netem_matches(output, settings):
                verified[client_id] = output
                print(f"Pumba verificado em {client_id}: {output.strip()}")
        if len(verified) == len(client_ids):
            return verified
        time.sleep(1)
    raise TimeoutError(f"Pumba nao aplicou as regras solicitadas em todos os clientes: {last_output}")


#### Execução do cenário

As funções desta seção acompanham os containers, recuperam as métricas e liberam os recursos ao final de cada simulação.


#### Leitura dos resultados

Lê os logs do Compose e verifica a saída do servidor, as rodadas e a participação registrada. Retorna métricas, estações e configuração em um resultado único, para impedir que uma execução incompleta seja apresentada como concluída.


In [ ]:
def _collect_scenario_results(*, name, requested_rounds, command, compose_config, started, num_clients, minimum, client_ids, allow_partial, min_fit_clients, round_timeout_seconds, settings, evidence):
    logs = subprocess.run(
        command + ["logs", "--no-color"],
        input=compose_config,
        cwd=TUTORIAL_DIR,
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    print(logs)
    inspect = subprocess.run(
        ["docker", "inspect", "--format", "{{.State.ExitCode}}", f"{PROJECT_NAME}-server"],
        capture_output=True,
        text=True,
        check=True,
    )
    if int(inspect.stdout.strip()) != 0:
        raise RuntimeError(f"{name}: o servidor terminou com erro. Consulte os logs acima.")

    rounds = [
        json.loads(line.split("ROUND_RESULT ", 1)[1])
        for line in logs.splitlines()
        if "ROUND_RESULT " in line
    ]
    if ([item["round"] for item in rounds] != list(range(1, requested_rounds + 1))
            or any(not minimum <= item["clients"] <= num_clients for item in rounds)
            or any(len(item["fit_participants"]) != item["clients"] for item in rounds)
            or any(set(item["fit_participants"]) | set(item["missing_clients"]) != set(client_ids)
                   for item in rounds)):
        raise RuntimeError(f"{name}: resultados incompletos para as rodadas/clientes solicitados.")
    return {
        "name": name,
        "stations": {cid: path.stem for cid, path in prepare_client_data(num_clients).items()},
        "elapsed_seconds": round(time.monotonic() - started, 2),
        "rounds": rounds,
        "num_clients": num_clients, "pumba": settings, "pumba_verified": evidence,
        "allow_partial": allow_partial, "min_fit_clients": minimum,
        "round_timeout_seconds": round_timeout_seconds,
        "evaluation_mode": "distributed_available" if allow_partial else "distributed_all",
    }

O programa espera a porta Flower estar pronta antes de iniciar os clientes. 
Acompanha conexoes e falhas de servidor/clientes, sem aguardar o limite total 
quando o minimo de participantes se torna impossivel. No modo parcial, todos 
conectam inicialmente; depois vale `min_fit_clients`. O Pumba so e ativado 
apos as conexoes e o treino aguarda a verificacao das regras. 
O bloco `finally` remove os servicos mesmo quando ocorre uma falha.


In [ ]:
def _scenario_progress(client_ids, minimum, requested_rounds):
    inspected = run_docker_command(
        ["docker", "inspect", "--format", "{{.Name}} {{json .State}}",
         f"{PROJECT_NAME}-server", *(f"{PROJECT_NAME}-{cid}" for cid in client_ids)],
        "Falha ao consultar o estado dos containers.", show_output=False,
    )
    states = {}
    for line in inspected.stdout.splitlines():
        container, state = line.split(" ", 1)
        states[container.removeprefix(f"/{PROJECT_NAME}-")] = json.loads(state)
    logs = run_docker_command(
        ["docker", "logs", f"{PROJECT_NAME}-server"],
        "Falha ao consultar o servidor Flower.", show_output=False,
    )
    server_logs = logs.stdout + logs.stderr
    server = states["server"]
    if ("[server] Training failed" in server_logs or server["Status"] == "dead"
            or (server["Status"] == "exited" and server["ExitCode"] != 0)):
        raise RuntimeError("O servidor Flower falhou. Consulte os logs abaixo.")
    if server["Status"] == "exited":
        return False, server_logs
    if server["Status"] != "running":
        raise RuntimeError(f"Estado inesperado do servidor: {server['Status']}.")

    rounds_complete = server_logs.count("ROUND_RESULT ") == requested_rounds
    failed = {
        cid: states[cid] for cid in client_ids
        if states[cid]["Status"] == "dead"
        or (states[cid]["Status"] == "exited"
            and (states[cid]["ExitCode"] != 0 or not rounds_complete))
    }
    required = minimum if "[server] All clients connected" in server_logs else len(client_ids)
    if len(client_ids) - len(failed) < required:
        details = ", ".join(f"{cid} (exit={state['ExitCode']})" for cid, state in failed.items())
        raise RuntimeError(f"Clientes encerrados antes de concluir o cenario: {details}. "
                           f"O minimo exigido e {required}. Consulte os logs abaixo.")
    return True, server_logs


In [ ]:
def _execute_scenario(*, name, requested_rounds, chaos, client_ids, settings, timeout_seconds, start_signal, command, compose_config, started, num_clients, minimum, allow_partial, min_fit_clients, round_timeout_seconds):
    try:
        deadline = time.monotonic() + timeout_seconds
        run_docker_command(
            command + ["up", "-d", "--wait", "--wait-timeout",
                       str(min(300, timeout_seconds)), "server"],
            "O servidor Flower nao ficou pronto para receber conexoes.",
            compose_config=compose_config,
        )
        run_docker_command(
            command + ["up", "-d", "--no-deps", *client_ids],
            "Falha ao iniciar os containers. Consulte a saida Docker acima.",
            compose_config=compose_config,
        )
        connection_deadline = min(deadline, time.monotonic() + 300)
        while True:
            running, startup_logs = _scenario_progress(client_ids, minimum, requested_rounds)
            if "[server] All clients connected" in startup_logs:
                break
            if not running:
                raise RuntimeError("O servidor terminou antes de conectar todos os clientes.")
            if time.monotonic() >= connection_deadline:
                raise TimeoutError("Clientes nao conectaram antes de iniciar o treino.")
            time.sleep(0.2)
        if chaos:
            run_docker_command(command + ["up", "-d", "--no-deps", "pumba"],
                               "Falha ao iniciar Pumba apos a conexao dos clientes.",
                               compose_config=compose_config)
        evidence = verify_pumba(settings["clients"], settings, min(deadline, time.monotonic() + 60)) if chaos else {}
        if chaos:
            start_signal.write_text("verified", encoding="utf-8")
        while time.monotonic() < deadline:
            if chaos:
                require_pumba_running()
            running, _ = _scenario_progress(client_ids, minimum, requested_rounds)
            if not running:
                break
            time.sleep(2)
        else:
            raise TimeoutError(f"{name}: tempo limite de {timeout_seconds}s excedido.")

        return _collect_scenario_results(
            name=name, requested_rounds=requested_rounds, command=command, compose_config=compose_config,
            started=started, num_clients=num_clients, minimum=minimum,
            client_ids=client_ids, allow_partial=allow_partial,
            min_fit_clients=min_fit_clients, round_timeout_seconds=round_timeout_seconds,
            settings=settings, evidence=evidence,
        )
    except (RuntimeError, TimeoutError):
        diagnostics = subprocess.run(command + ["logs", "--no-color"], cwd=TUTORIAL_DIR, capture_output=True, text=True, input=compose_config)
        print(diagnostics.stdout, diagnostics.stderr)
        raise
    finally:
        cleanup = subprocess.run(
            compose_command("chaos", include_stdin_config=True) + ["down", "--remove-orphans"],
            cwd=TUTORIAL_DIR,
            check=False,
            capture_output=True, text=True, input=compose_config,
        )
        if cleanup.returncode:
            print(f"Falha na limpeza Docker: {cleanup.stdout}{cleanup.stderr}")
        start_signal.unlink(missing_ok=True)

#### Chamada da execução

`run_scenario` é a entrada principal do experimento: prepara o manifesto, verifica o Docker e constrói a imagem antes da medição. Cria um diretório temporário de controle para o servidor e, com `collect_predictions=True`, diretórios separados para as previsões de cada cliente. O servidor não recebe os CSVs nem os pares real/previsto.


In [ ]:
def run_scenario(*, chaos=False, timeout_seconds=900, num_clients=3, rounds=5,
                 delay_time=0, delay_jitter=0, loss_percent=0, delay_distribution="normal",
                 interface="eth0", duration=None,
                 allow_partial=True, min_fit_clients=None,
                 round_timeout_seconds=20, chaos_clients=None, epochs=50, learning_rate=0.001,
                 training_mode="federated", model_kind="standard",
                 early_stopping=True, patience=2, min_delta=0.0001, collect_predictions=True,
                 delay_correlation=0, loss_correlation=0,
                 duplicate_percent=0, duplicate_correlation=0,
                 corrupt_percent=0, corrupt_correlation=0, rate_value=None,
                 rate_packet_overhead=0, rate_cell_size=0, rate_cell_overhead=0):
    requested_rounds = rounds
    name = (
        f"Pumba: delay={delay_time}ms jitter={delay_jitter}ms loss={loss_percent}%"
        if chaos else f"Baseline: {num_clients} clientes"
    )
    manifest, client_ids, settings = scenario_configuration(
        num_clients, rounds, chaos, timeout_seconds, delay_time, delay_jitter,
        loss_percent, delay_distribution, interface, duration,
        allow_partial, min_fit_clients, round_timeout_seconds, chaos_clients,
        epochs=epochs, learning_rate=learning_rate,
        training_mode=training_mode, model_kind=model_kind,
        early_stopping=early_stopping, patience=patience, min_delta=min_delta,
        collect_predictions=collect_predictions,
        delay_correlation=delay_correlation, loss_correlation=loss_correlation,
        duplicate_percent=duplicate_percent, duplicate_correlation=duplicate_correlation,
        corrupt_percent=corrupt_percent, corrupt_correlation=corrupt_correlation,
        rate_value=rate_value, rate_packet_overhead=rate_packet_overhead,
        rate_cell_size=rate_cell_size, rate_cell_overhead=rate_cell_overhead,
    )
    minimum = (1 if allow_partial else num_clients) if min_fit_clients is None else min_fit_clients
    name += f" | {'parcial' if allow_partial else 'completo'}, min={minimum}, timeout={round_timeout_seconds}s"
    run_docker_command(
        ["docker", "info"],
        "Docker Engine indisponivel. Abra o Docker Desktop, aguarde o engine iniciar "
        "e use Linux containers (WSL2 no Windows). Confirme com docker info antes de tentar novamente.",
    )
    run_docker_command(compose_command("chaos") + ["down", "--remove-orphans"], "Falha ao limpar o cenario anterior.")
    with tempfile.TemporaryDirectory(prefix="fl-pumba-") as directory:
        data_dir = Path(directory) / "control"
        data_dir.mkdir()
        prediction_dir = Path(directory) / "predictions"
        if collect_predictions:
            for client_id in client_ids:
                client_dir = prediction_dir / client_id
                client_dir.mkdir(parents=True)
                manifest["services"][client_id]["volumes"].append({
                    "type": "bind", "source": str(client_dir.resolve()),
                    "target": "/predictions", "read_only": False,
                })
        data_mount = {"type": "bind", "source": str(data_dir.resolve()),
                      "target": "/data", "read_only": True}
        manifest["services"]["server"].setdefault("volumes", []).append(data_mount)
        start_signal = data_dir / "start-training.flag"
        start_signal.unlink(missing_ok=True)
        compose_config = json.dumps(manifest, indent=2)
        command = compose_command("chaos" if chaos else None, include_stdin_config=True)
        if chaos:
            run_docker_command(["docker", "pull", TC_IMAGE], "Falha ao obter a imagem tc para Pumba.")
        run_docker_command(
            command + ["--progress", "plain", "build"],
            "Falha ao construir a imagem Flower. Consulte a saida Docker acima.",
            compose_config=compose_config,
        )
        started = time.monotonic()
        result = _execute_scenario(
            name=name, requested_rounds=requested_rounds, chaos=chaos,
            client_ids=client_ids, settings=settings, timeout_seconds=timeout_seconds,
            start_signal=start_signal, command=command,
            compose_config=compose_config, started=started,
            num_clients=num_clients, minimum=minimum, allow_partial=allow_partial,
            min_fit_clients=min_fit_clients, round_timeout_seconds=round_timeout_seconds,
        )
        result["collect_predictions"] = collect_predictions
        if collect_predictions:
            result["predictions"] = {
                "round": requested_rounds, "targets": list(config["targets"]),
                "stations": {
                    client_id: json.loads((prediction_dir / client_id / "final.json").read_text(encoding="utf-8"))
                    for client_id in result["rounds"][-1]["evaluation_participants"]
                },
            }
            prediction_points(result)
        result["early_stopping"] = early_stopping
        result["patience"] = patience
        result["min_delta"] = min_delta
        result["epochs"] = epochs
        result["learning_rate"] = learning_rate
        result["training_mode"] = training_mode
        result["model_kind"] = model_kind
        return result

### Configuracao e execucao dos experimentos

Define seis clientes e cinco rodadas por padrao. `run_fl` usa cinquenta epocas
locais no maximo e taxa 0.001, com early stopping (`patience=2`, `min_delta=0.0001`).
Aceita alteracoes por argumento e guarda cada resultado
em `experiment_runs`. Inicialize esse dicionario somente ao comecar uma nova sessao.

In [ ]:
EXPERIMENT = dict(num_clients=6, rounds=5, allow_partial=True)
experiment_runs = {}

`run_fl` recebe opções de configuração, com cinco rodadas, `epochs=50` e `learning_rate=0.001` por padrão, e chama `run_scenario` uma vez. Por exemplo, `run_fl(epochs=2, learning_rate=0.0003)` substitui os parâmetros de otimização. A taxa deve ser positiva e finita. Após o sucesso, guarda o resultado com um identificador automático, como `Experimento 1`. Cada chamada preserva os resultados anteriores.


In [ ]:
def run_fl(*, epochs=50, learning_rate=0.001, early_stopping=True,
                   patience=2, min_delta=0.0001, collect_predictions=True, **settings):
    result = run_scenario(**{
        **EXPERIMENT, **settings, "epochs": epochs, "learning_rate": learning_rate,
        "early_stopping": early_stopping, "patience": patience, "min_delta": min_delta,
        "collect_predictions": collect_predictions,
    })
    number = len(experiment_runs) + 1
    while f"Experimento {number}" in experiment_runs:
        number += 1
    experiment_id = f"Experimento {number}"
    experiment_runs[experiment_id] = result
    return result

`export_results` salva explicitamente o dicionário de resultados em JSON. É uma opção para preservar o experimento antes de encerrar ou reiniciar o kernel.


In [ ]:
def export_results(path):
    """Grave os resultados somente quando quiser preservar o experimento."""
    path = Path(path)
    path.write_text(json.dumps(experiment_runs, indent=2), encoding="utf-8")
    print(f"Resultados exportados: {path.resolve()}")
    return path

### Metricas e apresentacao

As funcoes abaixo preparam resumos, tabelas compactas e graficos.
Cada hands-on chama essas funcoes apenas com os resultados do seu proprio caso.

### Estatísticas das execuções

`summarize_run` extrai as métricas da última rodada, o tempo acumulado e a participação por rodada. Organiza os indicadores de uma única execução, sem estatísticas de repetições.


In [ ]:
def summarize_run(run):
    summary = {}
    for label, key in (("RMSE", "test_rmse"), ("MAE", "test_mae"), ("R2", "test_r2"),
                       ("RMSE PM2.5", "test_rmse_pm25"), ("MAE PM2.5", "test_mae_pm25"), ("R2 PM2.5", "test_r2_pm25"),
                       ("RMSE PM10", "test_rmse_pm10"), ("MAE PM10", "test_mae_pm10"), ("R2 PM10", "test_r2_pm10")):
        summary[label] = run["rounds"][-1][key]
    summary["Tempo total treino (s)"] = run["rounds"][-1]["cumulative_seconds"]
    summary["Primeira rodada (s)"] = run["rounds"][0]["duration_seconds"]
    summary["Clientes por rodada"] = [item["clients"] for item in run["rounds"]]
    return summary

### Tabela de resultados

Importa os recursos de apresentação HTML do notebook e o escape de texto. Eles permitem exibir a comparação dos experimentos em uma tabela legível.


In [ ]:
from html import escape
from IPython.display import HTML, display

`html_table` transforma cabeçalhos e linhas em HTML, escapando o conteúdo textual. A rolagem horizontal mantém a tabela utilizável mesmo com métricas para os dois poluentes.


In [ ]:
def html_table(headers, rows):
    heading = ''.join(f'<th>{escape(str(value))}</th>' for value in headers)
    body = ''.join('<tr>' + ''.join(
        f'<td>{escape(str(value))}</td>' for value in row
    ) + '</tr>' for row in rows)
    return f'<div style="overflow-x:auto"><table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table></div>'

`configuration_values` reúne os parâmetros de rede, quantidade de clientes e política de participação de um resultado. Esses valores identificam a configuração que produziu as métricas apresentadas.


In [ ]:
def configuration_values(run):
    settings = run['pumba']
    return (
        settings['delay_time'] if settings else 0,
        settings['delay_jitter'] if settings else 0,
        settings['loss_percent'] if settings else 0,
        settings['delay_distribution'] if settings and settings['delay_distribution'] else '-',
        ', '.join(settings['clients']) if settings else '-',
        run['num_clients'], len(run['rounds']),
        'parcial' if run['allow_partial'] else 'completo',
        run['min_fit_clients'], run['round_timeout_seconds'],
    )

`render_results` monta a tabela do caso: clientes, rede, RMSE, MAE, R2 e tempo
das rodadas. Configuracoes e resultados por poluente ficam recolhidos.

In [ ]:
def render_results(experiments):
    if not experiments or any(not run.get('rounds') for run in experiments.values()):
        raise ValueError('Nao ha resultados completos para comparar.')
    combination_rows = []
    details = []
    for label, run in experiments.items():
        final = run['rounds'][-1]
        delay, jitter, loss, *other_settings = configuration_values(run)
        network = f'{delay:g} ms / {jitter:g} ms / {loss:g}%' if run['pumba'] else 'Sem caos'
        combination_rows.append([
            label, run['num_clients'], network,
            f"{final['test_rmse']:.2f}", f"{final['test_mae']:.2f}",
            f"{final['test_r2']:.3f}", f"{final['cumulative_seconds']:.2f}",
        ])
        pollutant_rows = [
            [target, *[f"{final[f'test_{metric}_{suffix}']:.3f}" for metric in ('rmse', 'mae', 'r2')]]
            for target, suffix in (('PM2.5', 'pm25'), ('PM10', 'pm10'))
        ]
        configuration_rows = [
            ['Estacoes', ', '.join(run['stations'].values())],
            ['Limite de epocas por rodada', 0 if run.get('training_mode') == 'persistence' else run['epochs']],
            ['Early stopping', 'Nao se aplica' if run.get('training_mode') == 'persistence'
             else run['early_stopping']],
            ['Patience / min_delta', f"{run['patience']} / {run['min_delta']:g}"],
            ['Duplicacao (% / correlacao)', f"{run['pumba']['duplicate_percent']:g} / {run['pumba']['duplicate_correlation']:g}" if run['pumba'] else '-'],
            ['Corrupcao (% / correlacao)', f"{run['pumba']['corrupt_percent']:g} / {run['pumba']['corrupt_correlation']:g}" if run['pumba'] else '-'],
            ['Taxa / overhead / cell size', f"{run['pumba']['rate_value']} / {run['pumba']['rate_packet_overhead']} / {run['pumba']['rate_cell_size']}" if run['pumba'] else '-'],
            ['Taxa de aprendizado', run.get('learning_rate', 'Nao registrada')],
            ['Modo', run.get('training_mode', 'federated')],
            ['Modelo', run.get('model_kind', 'standard')],
            ['MSE global', f"{final['test_loss']:.3f}"],
            *zip(['Distribuicao do jitter', 'Clientes com caos', 'Clientes', 'Rodadas',
                  'Participacao', 'Minimo de respostas', 'Timeout da rodada (s)'], other_settings),
        ]
        work_rows = [
            [row["round"], cid, work["epochs"], work["selected_epoch"],
             "Sim" if work["early_stopped"] else "Nao", work["optimizer_steps"]]
            for row in run["rounds"] for cid, work in row["training_work"].items()
        ]
        details.append(
            f'<details><summary>{escape(str(label))}</summary>'
            + html_table(['Poluente', 'RMSE', 'MAE', 'R2'], pollutant_rows)
            + html_table(['Configuracao', 'Valor'], configuration_rows)
            + html_table(['Rodada', 'Cliente', 'Epocas executadas', 'Epoca retornada',
                          'Parada antecipada', 'Atualizacoes'], work_rows)
            + '</details>'
        )
    return (
        '<h3>Comparacao dos experimentos</h3>'
        + '<p>Metricas finais; tempo acumulado das rodadas, incluindo avaliacao. '
          'Rede: atraso / jitter / perda.</p>'
        + html_table(['Experimento', 'Clientes', 'Rede', 'RMSE', 'MAE', 'R2', 'Tempo (s)'], combination_rows)
        + '<h4>Detalhes por experimento</h4>' + ''.join(details)
    )

`plot_results` mostra o RMSE e o MAE de teste por rodada.
O tempo e as outras metricas permanecem na tabela.


In [ ]:
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt

Cada curva corresponde a uma execucao do caso atual. RMSE e MAE de teste sao
mostrados separadamente; o teste nao deve orientar a escolha de hiperparametros.

In [ ]:
def plot_results(experiments):
    if not experiments or any(not run.get("rounds") for run in experiments.values()):
        raise ValueError("Nao ha resultados completos para plotar.")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for label, run in experiments.items():
        for axis, key, title in (
            (axes[0], "test_rmse", "RMSE de teste"),
            (axes[1], "test_mae", "MAE de teste"),
        ):
            axis.plot([row["round"] for row in run["rounds"]],
                      [row[key] for row in run["rounds"]], marker="o", label=label)
            axis.set_title(title)
            axis.set_xlabel("Rodada")
            axis.set_ylabel("RMSE" if key == "test_rmse" else "MAE")
            axis.grid(alpha=0.25)
    for axis in axes:
        axis.legend(fontsize="small")
    fig.tight_layout()
    plt.show()

### Grafico real x previsto

Por padrao, cada cliente grava os pares do teste da ultima rodada
em uma pasta temporaria exclusiva. O notebook coleta os pares antes de remover a pasta;
o servidor Flower nao recebe esses arquivos. Os dados ficam no resultado e serao
incluidos por `export_results`. Com `collect_predictions=False`, sao coletadas apenas metricas.

`plot_predictions` usa os residuos assinados (`previsto - real`) de todos os pares
para separar as caudas de 5% e os 90% centrais, independentemente por poluente.
Empates sao resolvidos pela ordem dos pares; percentuais sao aproximados por arredondamento.
Os pontos vermelhos/azuis sao as caudas, nao necessariamente erros positivos/negativos.
Acima de `y=x` ha superestimacao; abaixo, subestimacao. Verde nao significa erro aceitavel.
O grafico mostra ate 2.000 pontos por painel, com amostragem deterministica estratificada;
as metricas continuam usando todos os pares. Cada par representa uma janela/horizonte:
a mesma hora pode aparecer em previsoes de horizontes diferentes.


In [ ]:
def prediction_points(run):
    if not run.get("rounds") or not run.get("predictions"):
        raise ValueError("Execute run_fl() com a coleta de previsoes padrao para obter os pares real/previsto.")
    data, final = run["predictions"], run["rounds"][-1]
    if (data["round"] != final["round"] or data["targets"] != ["PM2.5", "PM10"]
            or set(data["stations"]) != set(final["evaluation_participants"])):
        raise ValueError("Previsoes incompativeis com a ultima rodada ou seus participantes.")
    actual, predicted = [], []
    for client_id, values in sorted(data["stations"].items()):
        truth = np.asarray(values["actual"], dtype=float)
        forecast = np.asarray(values["predicted"], dtype=float)
        if (truth.ndim != 2 or truth.shape[1] != 2 or len(truth) < 20
                or forecast.shape != truth.shape
                or not np.isfinite(truth).all() or not np.isfinite(forecast).all()):
            raise ValueError(f"Pares real/previsto invalidos: {client_id}.")
        actual.append(truth)
        predicted.append(forecast)
    return np.concatenate(actual), np.concatenate(predicted)

In [ ]:
def plot_predictions(run, title="Real x previsto", max_points=2000):
    if isinstance(max_points, bool) or not isinstance(max_points, int) or max_points < 20:
        raise ValueError("max_points deve ser um inteiro maior ou igual a 20.")
    actual, predicted = prediction_points(run)
    total = len(actual)
    tail_size = total // 20
    shown = min(total, max_points)
    shown_tail = shown // 20
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for target_index, (axis, target) in enumerate(zip(axes, run["predictions"]["targets"])):
        truth, forecast = actual[:, target_index], predicted[:, target_index]
        order = np.argsort(forecast - truth, kind="stable")
        groups = (
            (order[tail_size:-tail_size], shown - 2 * shown_tail, "#00b050", "90% centrais"),
            (order[:tail_size], shown_tail, "#2166ac", "5% menores residuos"),
            (order[-tail_size:], shown_tail, "#e60000", "5% maiores residuos"),
        )
        rng = np.random.default_rng(42)
        for indices, count, color, label in groups:
            selected = indices if len(indices) == count else rng.choice(indices, size=count, replace=False)
            axis.scatter(truth[selected], forecast[selected], s=13, alpha=0.65,
                         color=color, label=label, edgecolors="none")
        lower = min(truth.min(), forecast.min())
        upper = max(truth.max(), forecast.max())
        padding = max((upper - lower) * 0.03, 1.0)
        limits = (lower - padding, upper + padding)
        axis.plot(limits, limits, color="gray", linewidth=1, linestyle="--", label="Ideal: y = x")
        axis.set(xlim=limits, ylim=limits, xlabel="Valor real", ylabel="Valor previsto",
                 title=f"{target} | rodada {run['predictions']['round']}")
        axis.set_aspect("equal", adjustable="box")
        axis.grid(alpha=0.2)
        axis.legend(fontsize="small")
    fig.suptitle(f"{title}\nTeste: {shown:,} de {total:,} pares por poluente (unidades originais)")
    fig.tight_layout()
    plt.show()
    return fig


## 2. Hands-on

### Como usar `run_fl`

A funcao `run_fl` sera usada para a experimentacao nos exercicios abaixo.
Cada chamada executa um cenario de aprendizado federado, retorna um dicionario
com configuracao e metricas por rodada e registra o resultado em `experiment_runs`.
Os argumentos sao passados pelo nome; os omitidos usam os valores padrao.

| Parametro | Padrao | Uso |
|---|---|---|
| `num_clients` | `6` | Quantidade de estacoes/clientes, entre 1 e 6. |
| `rounds` | `5` | Numero de rodadas federadas. |
| `epochs` | `50` | Maximo de epocas locais por rodada. |
| `learning_rate` | `0.001` | Taxa de aprendizado do Adam. |
| `early_stopping` | `True` | Interrompe epocas quando a perda de treino deixa de melhorar. |
| `patience` / `min_delta` | `2` / `0.0001` | Epocas sem melhora significativa e reducao minima da perda. |
| `collect_predictions` | `True` | Coleta localmente os pares de teste da ultima rodada para o grafico real x previsto. |
| `chaos` | `False` | Ativa as condicoes de rede do Pumba. |
| `delay_time` / `delay_jitter` / `loss_percent` | `0` / `0` / `0` | Atraso, variacao do atraso e perda percentual, usados com `chaos=True`. |
| `duplicate_percent` / `corrupt_percent` | `0` / `0` | Percentual de duplicacao ou corrupcao; afeta a saida. |
| `rate_value` | `None` | Habilita o limite de banda, por exemplo `"100kbit"`. |
| `delay_distribution` | `"normal"` | Distribuicao do jitter. |
| `interface` / `duration` | `"eth0"` / `None` | Interface afetada e duracao em segundos; `None` usa timeout global + 60 s. |
| `chaos_clients` | `None` | Lista dos clientes afetados; `None` afeta todos. |
| `round_timeout_seconds` | `20` | Espera por respostas nas etapas de treino e avaliacao, comum aos clientes. |
| `allow_partial` / `min_fit_clients` | `True` / `1` | Aceita qualquer quantidade positiva de respostas no treino e avaliacao; zero gera erro. `False` exige todos por padrao. |
| `timeout_seconds` | `900` | Limite global do cenario, nao um timeout individual de cliente. |

Exemplo de uso (a execucao dos exercicios comeca no baseline):

```python
resultado = run_fl(num_clients=4)
plot_predictions(resultado, title="Exemplo com quatro estacoes")
```

Cada caso contem **implementacao, execucao e suas proprias metricas**.
Execute o baseline primeiro: o caso de rede o reutiliza como referencia.
Altere os parametros indicados em cada exercicio e consulte suas tabelas e graficos.
Nos casos 2.1, 2.2 e 2.3, cada exercicio tem enunciado e resolucao executavel.
As perguntas de analise ficam na parte 3.


### 2.1 Baseline e numero de rodadas

**Exemplo simples:** execute o FL sem caos com seis clientes e cinco rodadas. Em seguida, visualize
o valor real x previsto no teste da ultima rodada.


In [ ]:
baseline = run_fl()
plot_predictions(baseline, title="Baseline: 5 rodadas");

**Metricas do exemplo:** a tabela resume a ultima rodada; os graficos abaixo
mostram a evolucao de RMSE e MAE. O grafico real x previsto aparece junto da execucao.


In [ ]:
baseline_results = {"Baseline": baseline}
display(HTML(render_results(baseline_results)))
plot_results(baseline_results)

#### Exercicios: 3, 8 e 13 rodadas

Execute tres experimentos independentes, alterando somente `rounds` para **3, 8 e 13**.
Mantenha seis clientes, ausencia de caos, ate 50 epocas e os mesmos parametros de early stopping.
Em cada execucao, plote um grafico real x previsto usando a coleta de previsoes padrao.
Use variaveis diferentes para preservar cada resultado.

**Atencao:** alterar as rodadas tambem altera os blocos de treino e o numero de janelas
nas fronteiras. Nao e uma comparacao apenas de mais atualizacoes sobre dados identicos.


**Exercicio — 3 rodadas:** execute o FL com `rounds=3` e, logo depois,
plote o grafico do resultado com um titulo que identifique as 3 rodadas.

**Resolucao:**


In [ ]:
result_3_rounds = run_fl(rounds=3)
plot_predictions(result_3_rounds, title="FL: 3 rodadas");

**Exercicio — 8 rodadas:** execute o FL com `rounds=8` e, logo depois,
plote o grafico do resultado com um titulo que identifique as 8 rodadas.

**Resolucao:**


**Exercicio — 13 rodadas:** execute o FL com `rounds=13` e, logo depois,
plote o grafico do resultado com um titulo que identifique as 13 rodadas.

**Resolucao:**


#### Exercicio final: tabela comparativa

Reuna os resultados das execucoes de **3, 8 e 13 rodadas** em uma unica tabela.
Identifique cada execucao e apresente RMSE, MAE, R2 e tempo acumulado das rodadas.
Use as metricas da ultima rodada de cada execucao e reutilize os resultados
armazenados, sem executar o FL novamente. Utilize `render_results` para montar
a tabela e `display(HTML(...))` para exibi-la.

**Resolucao:**


### 2.2 Condicoes de rede

**Exemplo simples:** mantenha os parametros do baseline e aplique 250 ms de atraso,
sem jitter nem perda. Execute uma vez e plote o valor real x previsto.
Depois do exemplo, resolva os exercicios de atraso, jitter, perda e caos seletivo.


In [ ]:
network_run = run_fl(
    chaos=True, delay_time=250,
)
plot_predictions(network_run, title="FL com atraso de 250 ms");

**Metricas da rede:** compara o cenario escolhido com o baseline sem caos.
As evidencias das regras do Pumba ficam no resultado do cenario.

In [ ]:
network_results = {"Sem caos": baseline, "Atraso de 250 ms": network_run}
display(HTML(render_results(network_results)))
plot_results(network_results)

#### Exercicios: parametros de caos do Pumba


**Exercicio 1 - Atraso:** execute com `delay_time=250`, sem jitter nem perda,
afetando todos os clientes. Guarde o resultado e plote o grafico.

**Resolucao:**


**Exercicio 2 - Atraso e jitter:** execute com atraso de 250 ms, jitter de
100 ms e distribuicao normal, sem perda. Guarde o resultado e plote o grafico.

**Resolucao:**


**Exercicio 3 - Perda de pacotes:** aplique perda de **70%** nos pacotes de saida
somente em `client-6`. Guarde o resultado e plote o grafico real x previsto.
A perda maior aumenta a interferencia, mas nao garante timeout; confirme ausencias
em `missing_clients` e prazo excedido por `DEADLINE_EXCEEDED` nos logs.
Use participacao parcial com minimo de uma resposta e prazo de 20 segundos:
atualizacoes que nao chegam no prazo ficam fora da agregacao. A avaliacao tambem
exige uma resposta; confira `evaluation_participants`, pois as estacoes avaliadas
podem diferir do baseline e impedir uma comparacao direta das metricas.

**Resolucao:**


**Exercicio 4 - Duplicacao:** duplique 10% dos pacotes de saida nos clientes
5 e 6, sem atraso, jitter, perda ou corrupcao. Plote o grafico real x previsto.

**Resolucao:**


**Exercicio 5 - Corrupcao:** corrompa 10% dos pacotes de saida nos clientes
5 e 6, sem atraso, jitter, perda ou duplicacao. Plote o grafico real x previsto.

**Resolucao:**


**Exercicio 6 - Limitacao de banda:** limite a taxa de saida a `10kbit`
no cliente 6, sem atraso, jitter, perda, duplicacao ou corrupcao.
Use participacao parcial com minimo de uma resposta e prazo de 20 segundos.
A banda menor nao garante timeout; confira `missing_clients` e os logs.
Confira tambem `evaluation_participants` antes de comparar as metricas com o baseline.
Plote o grafico real x previsto.

**Resolucao:**


**Exercicio 7 - Tabela comparativa:** apresente uma unica tabela com o baseline
sem caos e os seis cenarios anteriores: atraso, atraso com jitter, perda de 70%,
duplicacao, corrupcao e limitacao de banda.
Compare RMSE, MAE, R2 e tempo acumulado usando a ultima rodada de cada execucao.
Reutilize os resultados armazenados, sem executar o FL novamente.

**Resolucao:**


### 2.3 Numero de estacoes

**Exemplo simples:** execute o FL com duas estacoes, mantendo cinco rodadas,
ate 50 epocas, early stopping, taxa 0.001 e ausencia de caos. Plote o resultado.
Mudar as estacoes tambem muda a populacao avaliada.
Depois do exemplo, execute com 3, 4, 5 e 6 clientes e compare os resultados.


In [ ]:
station_run = run_fl(num_clients=2)
plot_predictions(station_run, title="FL com duas estacoes");

**Metricas por quantidade de estacoes:** apresenta apenas as execucoes deste caso.
Os detalhes recolhiveis identificam as estacoes de cada configuracao.

In [ ]:
station_results = {"2 estacoes": station_run}
display(HTML(render_results(station_results)))
plot_results(station_results)

#### Exercicios: 3, 4, 5 e 6 clientes

Execute um cenario para cada quantidade de clientes, mantendo os demais parametros padrao.
Use a coleta de previsoes padrao e plote um grafico real x previsto
logo apos cada execucao. Guarde os quatro resultados para a tabela final.
Seis clientes e o padrao de `run_fl`, portanto omita `num_clients` nessa chamada.
Adicionar estacoes tambem altera os dados de treino e a populacao de teste;
a comparacao nao isola somente o efeito da quantidade de clientes.
As celulas abaixo apresentam as resolucoes executaveis.


**Exercicio 1 - 3 clientes:** execute o FL com tres clientes e plote o grafico real x previsto.

**Resolucao:**


**Exercicio 2 - 4 clientes:** execute o FL com quatro clientes e plote o grafico real x previsto.

**Resolucao:**


**Exercicio 3 - 5 clientes:** execute o FL com cinco clientes e plote o grafico real x previsto.

**Resolucao:**


**Exercicio 4 - 6 clientes:** execute o FL com seis clientes e plote o grafico real x previsto.

**Resolucao:**


**Exercicio 5 - Tabela comparativa:** apresente uma tabela com os resultados de
3, 4, 5 e 6 clientes. Compare RMSE, MAE, R2 e tempo acumulado na ultima rodada.
Reutilize os quatro resultados armazenados, sem executar o FL novamente.

**Resolucao:**


## 3. Discussao e conclusoes

Use as metricas de cada hands-on para responder, sem executar novos cenarios aqui.

1. **Rodadas (2.1):** como os graficos de 3, 8 e 13 rodadas se aproximam da linha `y=x`?
   Mais rodadas melhoraram PM2.5 e PM10 da mesma forma?
2. **Condicoes de rede (2.2):** cada efeito do Pumba alterou mais o tempo ou as previsoes?
   Os clientes 5 e 6 ficaram fora de alguma rodada (`missing_clients`) e isso mudou as metricas?
3. **Numero de estacoes (2.3):** mais clientes melhoraram os resultados?
   Como o numero de estacoes afetou o tempo de cada rodada?


## 4. Limpeza

Remove os containers e a rede exclusivos do programa, sem apagar os CSVs.
Cada execucao ja realiza sua limpeza; esta celula tambem atende interrupcoes.
Para preservar resultados antes de reiniciar o kernel, use explicitamente
`export_results(TUTORIAL_DIR / "results.json")`.

In [ ]:
run_docker_command(
    compose_command("chaos") + ["down", "--remove-orphans"],
    "Falha ao limpar os containers do tutorial.",
)
print("Containers da pasta tutorial-fl-pumba removidos; os CSVs distribuídos foram preservados.")